In [3]:
import numpy as np
from scipy import sparse  
from numba import njit
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib
font = {'size' : 15}
matplotlib.rc('font', **font)
import pandas as pd
from skimage import measure
from scipy.stats import binned_statistic
from scipy.stats import linregress

### LOADING

In [4]:
field = 0.16
file_name = "data/file_field%1.2f.csv" % field

In [5]:
data = np.loadtxt(file_name, delimiter=',').astype('int')

### FUNCTIONS

In [6]:
def segment_noevents(data, no_event_val=-1):
    no_event_regions = [ r.astype('int') for r in measure.find_contours(data,no_event_val)]
    no_event_regions_sizes = np.array([r.shape[0] for r in no_event_regions])
    sorted_regions_idx = np.argsort(no_event_regions_sizes)
    inner_region = no_event_regions[sorted_regions_idx[-2]]
    outer_region = no_event_regions[sorted_regions_idx[-1]]
    center = inner_region.mean(axis=0)
    return inner_region,  outer_region, center

In [7]:
@njit
def data_to_list(image, value_t_skip = -1):
    time_values = np.unique(image)
    x_t = []
    y_t = []
    t_valid = []
    for t in time_values:
        if(t == value_t_skip):
            continue
        t_valid.append(t)
        indices_t = np.argwhere(image == t)
        x_t.append(indices_t[:,0])
        y_t.append(indices_t[:,1])
    return np.array(t_valid), x_t, y_t

In [8]:
def merge_list(time_values, x_values, y_values):
    lengths = np.array([len(x) for x in x_values])
    max_length = np.max(lengths)
    xy = -np.ones((len(lengths),max_length, 2), dtype=int)
    for idx in range(len(lengths)):
        xy[idx,:lengths[idx], 0] = x_values[idx]
        xy[idx,:lengths[idx], 1] = y_values[idx]
    return lengths, xy 

In [9]:
@njit
def get_edge_list(time_values, xy_values, lengths, maximal_distance, maximal_time):
    edge_list = []
    time_gaps = []
    spatial_gaps = []
    for idx, t in enumerate(time_values):
        length_t = lengths[idx]
        x_t, y_t =  xy_values[idx][:length_t,0], xy_values[idx][:length_t,1]
        for internal_idx,s in enumerate(time_values[idx+1:]):
            if((s-t) > maximal_time):
                break
            jdx = internal_idx + idx + 1
            length_s = lengths[jdx]
            x_s, y_s =  xy_values[jdx][:length_s,0], xy_values[jdx][:length_s,1]
            dists = np.abs(x_t[None,:] - x_s[:,None]) + np.abs(y_t[None,:] - y_s[:,None]) 
            min_dist = np.min(dists)
            if(min_dist <= maximal_distance):
                edge_list.append((idx, jdx))
                time_gaps.append(s-t)
                spatial_gaps.append(min_dist)
    return np.array(edge_list), np.array(time_gaps), np.array(spatial_gaps)

In [10]:
@njit
def data_polar(center, time_values, xy_values, lengths):
    polar = np.ones(xy_values.shape)*np.nan
    for idx, t in enumerate(time_values):
        length_t = lengths[idx]
        x_t, y_t =  xy_values[idx][:length_t,0], xy_values[idx][:length_t,1]
        polar[idx, :length_t,0] =np.sqrt((x_t-center[0])**2 + (y_t-center[1])**2)
        polar[idx,:length_t, 1]= np.arctan2(x_t-center[0], y_t-center[1])
    return polar

### EXECUTION

In [11]:
time_values, x_values, y_values  = data_to_list(data)

In [12]:
inner_region,  outer_region, center = segment_noevents(data)

In [13]:
lengths, xy_values  = merge_list(time_values, x_values, y_values)

In [14]:
polar = data_polar(center, time_values, xy_values.astype('float'), lengths)

In [15]:
dth = (2*np.pi)/2000
polar_df = pd.DataFrame()
polar_df['time'] = np.repeat(time_values, lengths)
polar_df['r'] = np.concatenate([ p[:l,0]  for l, p in zip(lengths, polar)])
polar_df['th'] = np.concatenate([ p[:l,1]  for l, p in zip(lengths, polar)])
polar_df['thi'] = np.floor(polar_df['th']/dth).astype('int')

In [16]:
radii_to_avg= []
vels_to_avg  = []
times_to_avg = []
for _, rows in polar_df.groupby('thi'):
    if(_ % 10 != 0):
        continue
    r_t = rows[['time','r']].groupby('time').max()
    v_t = np.diff(r_t.r)/np.diff(r_t.index)
    mask = v_t>0
    radii_to_avg.append(((r_t.r.values[:-1]+r_t.r.values[1:])/2)[mask])
    vels_to_avg.append(v_t[mask])
    times_to_avg.append((r_t.index.values[1:])[mask])
    #log_t = np.log(r_t.index.values/r_t.index.values[0])
    #log_r = np.log(r_t.r)
radii_to_avg = np.concatenate(radii_to_avg)
vels_to_avg = np.concatenate(vels_to_avg)
times_to_avg = np.concatenate(times_to_avg)

In [28]:
maximal_distance  = 4
maximal_time = 50
edge_list, time_attr, spatial_attr = get_edge_list(time_values, xy_values, lengths, maximal_distance, maximal_time)

In [29]:
#dest_folder = "/data/biophys/schimmenti/earthquakes/MOKE_clusters/"
dest_folder = "clusters_data/"
import os
os.makedirs(dest_folder, exist_ok=True)
np.savez("%sclusters_field=%1.2f.npz" % (dest_folder,field), time_values=time_values, xy_values=xy_values, lengths=lengths, edge_list=edge_list, time_attr=time_attr, spatial_attr=spatial_attr)